# Regressão Linear do Zero

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.


Neste tutorial vamos implementar um modelo de regressão linear para prever preços de imóveis na Califórnia. Construímos do zero usando NumPy. É uma ótima abordagem para entender modelos baseados em regressão.

Ao terminar, espera-se conhecer os blocos básicos de um modelo de regressão linear. Também espera-se conhecer o pipeline de leitura e transformação de dados para fluxos de aprendizado de máquina.


In [ ]:
## Import the usual libraries
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# Importando o Conjunto de Dados

O conjunto real pode ser obtido pela função `fetch_california_housing`, que baixa os dados para nós.

O parâmetro `as_frame` retorna um DataFrame do pandas, útil para visualizar o conteúdo dos dados.


In [ ]:
# Fetch the data using sklearn function
bunch = fetch_california_housing(download_if_missing=True, as_frame=True)

# Load the dataframe and view
df = bunch.frame
df.head()

Neste conjunto, a variável-alvo é o valor mediano de casas em distritos da Califórnia, em centenas de milhares de dólares (\$100,000).

Podemos olhar mais de perto vários parâmetros estatísticos usando o pandas. `describe` ajuda nisso.


In [ ]:
df.describe()

Como dá pra ver, os dados em cada coluna estão em escalas diferentes. Por exemplo, o número médio de quartos é em torno de 1, e a população média é em torno de 1425.

De forma geral, modelos de ML não funcionam bem quando os dados estão em escalas distintas. Por isso normalizamos os dados no intervalo [-1, 1]. O módulo [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) ajuda nisso.

Os dados de treino sempre devem ser normalizados. Os dados de teste devem ser normalizados usando os parâmetros calculados sobre o treino.


In [ ]:
# !wget https://raw.githubusercontent.com/Ankit152/Fish-Market/main/Fish.csv
# import pandas as pd
# df  = pd.read_csv("Fish.csv")
# y = df['Weight']
# x = df[["Length1", "Length2", "Length3", "Height", "Width","Weight"]]

df = bunch.frame
x = df.iloc[:,:-1] # Select all the columns, except the last column
y = df.iloc[:,-1:] # Select the last column
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.33, random_state = 1)

input_scalar = StandardScaler()
output_scalar = StandardScaler()

x_train = input_scalar.fit_transform(x_train).T # Normalize train data
x_test = input_scalar.transform(x_test).T # Only transform test data using values of train data

y_train = output_scalar.fit_transform(y_train).reshape(-1)
y_test = output_scalar.transform(y_test).reshape(-1)

dataset_copy = [ x_train.copy(), x_test.copy(),  y_train.copy(),  y_test.copy()]

# Modelo de Regressão Linear

Definimos o modelo de regressão linear do zero.

Um modelo de regressão linear tem a forma:

$y = a_1 x_1 + a_2 x_2 + \dots + a_n x_n + a_{n+1}$

Que pode ser reescrito como uma multiplicação de matriz:

$y = w^T x$

onde

$w = [a_1, a_2, \dots, a_n, a_{n+1}]^T$

$x = [x_1, x_2, \dots, x_n, 1]^T$


In [ ]:
class LinearRegression():
  def __init__(self, dim, lr = 0.1):
    assert isinstance
    self.lr = lr
    self.w = np.zeros((dim))
    self.grads = {"dw": np.zeros((dim)) +5}

  def forward(self, x):
    y = self.w.T @ x
    return y
  
  def backward(self, x, y_hat, y):
    assert y_hat.shape == y.shape
    self.grads["dw"] = (1 / x.shape[1]) * ((y_hat - y) @ x.T).T
    assert self.grads["dw"].shape == self.w.shape
    
    # print(self.grads["dw"])

  def optimize(self):
    self.w = self.w - self.lr * self.grads["dw"]

# Perda

Para regressão linear, várias funções de perda como erro absoluto médio (MAE), erro quadrático médio (MSE) ou raiz do MSE podem ser usadas.

Neste exemplo vamos usar a perda MSE:

$\text{MSE} = \dfrac{1}{m} \sum_{i=1}^{m} (y_{\text{true}}^{(i)} - y_{\text{pred}}^{(i)})^2$

onde $i$ é a observação e $m$ é o total de observações.

Para garantir que o modelo está correto, a perda deve cair a cada época.


In [ ]:
num_epochs = 10000
train_loss_history = []
test_loss_history = []
w_history = []
dim = x_train.shape[0]
num_train = x_train.shape[1]
num_test = x_test.shape[1]


model = LinearRegression(dim = dim, lr = 0.1)
for i in range(num_epochs):
  y_hat = model.forward(x_train)
  train_loss = 1/(2 * num_train) * ((y_train - y_hat) ** 2).sum()

  w_history.append(model.w)
  model.backward(x_train,y_hat,y_train)
  model.optimize()

  y_hat = model.forward(x_test)
  test_loss = 1/(2 * num_test) * ((y_test - y_hat) ** 2).sum()

  train_loss_history.append(train_loss)
  test_loss_history.append(test_loss)

  if i % 20 == 0:
    print(f"Epoch {i} | Train Loss {train_loss} | Test Loss {test_loss}")

plt.plot(range(num_epochs), train_loss_history, label = "Training")
plt.plot(range(num_epochs), test_loss_history, label = "Test")
plt.legend()
plt.show()

# Resultados

Antes de ver os resultados, precisamos reverter as transformações aplicadas na variável de saída $y$.

O método `inverse_transform` do `StandardScaler` ajuda nisso.


In [ ]:
from sklearn.metrics import mean_squared_error
y_test = output_scalar.inverse_transform(y_test[np.newaxis,:])
y_hat  = output_scalar.inverse_transform(y_hat[np.newaxis,:])
error = (((y_test - y_hat) ** 2).sum() / num_test )
print("Test Set Error", error)

# Bibliotecas

Em vez de codar tudo do zero (modelo, perdas, gradientes), existem várias bibliotecas que já implementam muitos algoritmos.

Elas costumam ser mais rápidas e otimizadas. Podemos usar `LinearRegression` e `SGDRegressor` do scikit-learn para comparar com o nosso modelo.


In [ ]:
from sklearn.linear_model import SGDRegressor


x_train, x_test, y_train, y_test = dataset_copy
sgd = SGDRegressor()
sgd.fit(x_train.T, y_train)
y_hat = sgd.predict(x_test.T)
y_test = output_scalar.inverse_transform(y_test[np.newaxis,:])
y_hat  = output_scalar.inverse_transform(y_hat[np.newaxis,:])
error = mean_squared_error(y_test, y_hat, squared = True)
print("Test Set Error", error)

In [ ]:
from sklearn.linear_model import LinearRegression as LR

x_train, x_test, y_train, y_test = dataset_copy
lr = LR()
lr.fit(x_train.T, y_train)
y_hat = lr.predict(x_test.T)
y_test = output_scalar.inverse_transform(y_test[np.newaxis,:])
y_hat  = output_scalar.inverse_transform(y_hat[np.newaxis,:])
error = mean_squared_error(y_test, y_hat, squared = True)
print("Test Set Error", error)

## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
